# Masa de acero con 4 resortes y campo magnético oscilante\n
\n
Planteamiento y resolución del problema de **oscilaciones rápidas** (método de Landau) para una masa en el centro, unida a 4 resortes (2 en $x$, 2 en $y$), considerando solo movimiento en $x$.

## 1. Modelo físico\n
\n
- Masa: $m$ (acero).\n
- Dos resortes activos en el eje $x$, cada uno con constante $k$ (los de $y$ no aportan fuerza si $y=0$).\n
- Rigidez efectiva en $x$: $k_{\mathrm{ef}} = 2k$.\n
- Campo magnético oscilante de alta frecuencia $\gamma$, con $\gamma \gg \omega_0$, donde $\omega_0^2 = k_{\mathrm{ef}}/m$.\n
\n
Modelamos la fuerza magnética rápida como\n
$$F_m(x,t)=f(x)\cos(\gamma t),\quad f(x)=q x,$$
donde $q$ resume el acoplamiento magneto-mecánico linealizado alrededor del equilibrio.\n
\n
La ecuación completa queda:\n
$$m\ddot x = -k_{\mathrm{ef}}x + qx\cos(\gamma t).$$

## 2. Aproximación de oscilaciones rápidas (Landau)\n
\n
Escribimos $x(t)=X(t)+\xi(t)$ con:\n
- $X(t)$: movimiento lento.\n
- $\xi(t)$: corrección rápida de pequeña amplitud.\n
\n
Para $\gamma\gg\omega_0$, Landau muestra que el movimiento lento satisface\n
$$m\ddot X = -\frac{d}{dX}U_{\mathrm{ef}}(X),\quad U_{\mathrm{ef}}(X)=U_0(X)+\frac{f(X)^2}{4m\gamma^2},$$
con $U_0(X)=\tfrac12 k_{\mathrm{ef}}X^2$ y $f(X)=qX$.\n
\n
Entonces\n
$$U_{\mathrm{ef}}(X)=\frac12 k_{\mathrm{ef}}X^2 + \frac{q^2}{4m\gamma^2}X^2
=\frac12\left(k_{\mathrm{ef}}+\frac{q^2}{2m\gamma^2}\right)X^2.$$
\n
La frecuencia lenta efectiva es\n
$$\Omega^2 = \frac{k_{\mathrm{ef}}}{m} + \frac{q^2}{2m^2\gamma^2} = \omega_0^2 + \frac{q^2}{2m^2\gamma^2}. $$
\n
La oscilación rápida de primer orden es\n
$$\xi(t)\approx -\frac{f(X)}{m\gamma^2}\cos(\gamma t)= -\frac{qX}{m\gamma^2}\cos(\gamma t).$$

In [3]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

In [ ]:
# Parámetros de ejemplo (SI)\n
m = 0.50          # kg\n
k = 20.0          # N/m, cada resorte sobre x\n
k_eff = 2*k       # dos resortes en x\n
q = 8.0           # N/m, acoplamiento magnético linealizado\n
gamma = 80.0      # rad/s (alta frecuencia)\n
\
omega0 = np.sqrt(k_eff/m)\
Omega = np.sqrt(omega0**2 + q**2/(2*m**2*gamma**2))\n
\n
print(f"omega0 = {omega0:.5f} rad/s")\n
print(f"Omega  = {Omega:.5f} rad/s")\n
print(f"gamma/omega0 = {gamma/omega0:.2f}  (debe ser >> 1)")

In [ ]:
# Dinámica completa: m x'' = -k_eff x + q x cos(gamma t)\n
def full_system(t, y):\n
    x, v = y\n
    a = (-k_eff*x + q*x*np.cos(gamma*t)) / m\n
    return [v, a]\n
\n
# Dinámica promediada (lenta): X'' + Omega^2 X = 0\n
def avg_system(t, y):\n
    X, V = y\n
    A = -(Omega**2)*X\n
    return [V, A]\n
\n
x0 = 0.02     # m\n
v0 = 0.0      # m/s\n
t_end = 8.0   # s\n
t_eval = np.linspace(0, t_end, 12000)\n
\n
sol_full = solve_ivp(full_system, [0, t_end], [x0, v0], t_eval=t_eval, rtol=1e-8, atol=1e-10)\n
sol_avg = solve_ivp(avg_system, [0, t_end], [x0, v0], t_eval=t_eval, rtol=1e-10, atol=1e-12)

In [ ]:
t = sol_full.t\n
x_full = sol_full.y[0]\n
X_slow = sol_avg.y[0]\n
\n
# Corrección rápida de Landau usando X(t) lento\n
xi = -(q*X_slow)/(m*gamma**2) * np.cos(gamma*t)\n
x_landau = X_slow + xi\n
\n
plt.figure(figsize=(11, 5))\n
plt.plot(t, x_full, lw=1.1, label='Sistema completo x(t)')\n
plt.plot(t, X_slow, '--', lw=2.0, label='Promedio lento X(t)')\n
plt.plot(t, x_landau, lw=1.0, alpha=0.8, label='Aprox. Landau X+xi')\n
plt.xlabel('t [s]')\n
plt.ylabel('Desplazamiento [m]')\n
plt.title('Oscilaciones rápidas: comparación completo vs. promedio')\n
plt.grid(alpha=0.3)\n
plt.legend()\n
plt.tight_layout()\n
plt.show()

## 3. Conclusiones\n
\n
1. El problema se reduce a un oscilador armónico con una fuerza rápida proporcional a $x\cos(\gamma t)$.\n
2. Para $\gamma\gg\omega_0$, el efecto principal del campo rápido es un **endurecimiento efectivo** del potencial: aumenta la frecuencia lenta de $\omega_0$ a $\Omega$.\n
3. La trayectoria real es la suma de un movimiento lento $X(t)$ y una oscilación pequeña de alta frecuencia $\xi(t)$ alrededor de $X$.\n
4. La simulación confirma la validez de la aproximación de Landau cuando la separación de escalas temporales es grande.